# 3. Liquidity & TVL — QA

**Purpose** — Validate liquidity reconstruction, tick distribution, active vs out-of-range TVL split, and LP-flow consistency.

**Source tables**
- `mart_tvl_daily`
- `mart_fee_yield_daily`  # deprecated view → mart_pool_screener_daily
- `mart_tvl_distribution_daily_centered`
- `mart_tvl_distribution_daily_metrics`
- `mart_tvl_distribution_hourly_centered`
- `mart_tick_daily`
- `int_lp_macro_daily`
- `int_range_liquidity_daily`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


## §1 — Data fetch & coverage

In [3]:
tvl = db.read_query(f"""
SELECT pool, date, tvl_token0, tvl_token1, tvl_token0_equiv AS tvl_total_token0, price_token1_in_token0
FROM mart_tvl_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
ORDER BY date
""")
fy = db.read_query(f"""
SELECT pool, date, tvl_total_token0, tvl_active_token0, tvl_outofrange_token0, fees_token0_equiv, apr
FROM mart_fee_yield_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
ORDER BY date
""")
flow = db.read_query(f"""
SELECT pool, date, mint_count, burn_count, mint_token0, burn_token0, net_lp_flow_token0
FROM int_lp_macro_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
ORDER BY date
""")
print(f"tvl daily: {tvl.height}  rows  ({tvl['date'].min()} → {tvl['date'].max()})")
print(f"fee yield: {fy.height}  rows")
print(f"lp flow:   {flow.height} rows")

tvl daily: 1798  rows  (2021-05-05 → 2026-04-06)
fee yield: 1798  rows
lp flow:   1798 rows


## §2 — Sanity checks

In [4]:
# Range validity in int_range_liquidity_daily
bad_rng = db.read_query(f"""
SELECT countIf(tick_lower >= tick_upper) AS bad_range,
       countIf(net_liquidity < 0)         AS neg_liquidity,
       countIf(modulo(tick_lower, 10) != 0 OR modulo(tick_upper, 10) != 0) AS mis_aligned
FROM int_range_liquidity_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
""").row(0, named=True)
qa_check("int_range_liquidity_daily: tick_lower < tick_upper", int(bad_rng["bad_range"]) == 0, detail=str(bad_rng["bad_range"]))
qa_check("int_range_liquidity_daily: net_liquidity > 0",       int(bad_rng["neg_liquidity"]) == 0, detail=str(bad_rng["neg_liquidity"]))
qa_check("int_range_liquidity_daily: ticks aligned to spacing 10", int(bad_rng["mis_aligned"]) == 0, detail=str(bad_rng["mis_aligned"]))

🟢 PASS | int_range_liquidity_daily: tick_lower < tick_upper  —  0
🟢 PASS | int_range_liquidity_daily: net_liquidity > 0  —  0
🟢 PASS | int_range_liquidity_daily: ticks aligned to spacing 10  —  0


In [5]:
# TVL reconciliation: active + OOR ≈ total
recon = fy.with_columns(
    (pl.col("tvl_active_token0") + pl.col("tvl_outofrange_token0")).alias("computed_total"),
    (((pl.col("tvl_active_token0") + pl.col("tvl_outofrange_token0")) - pl.col("tvl_total_token0")).abs()
     / pl.col("tvl_total_token0").clip(1e-6, None)).alias("rel_resid"),
)
n_bad = recon.filter(pl.col("rel_resid") > 0.01).height
qa_check("TVL active + OOR ≈ total (±1%)",
         n_bad == 0, severity="warn", detail=f"days_off={n_bad}")

🟢 PASS | TVL active + OOR ≈ total (±1%)  —  days_off=0


In [6]:
delta = flow.join(tvl.select("date", "tvl_total_token0"), on="date").with_columns(
    (pl.col("tvl_total_token0") - pl.col("tvl_total_token0").shift(1)).alias("delta_tvl"),
    (pl.col("mint_token0") - pl.col("burn_token0")).alias("net_flow"),
)
# Only check direction consistency on days with material flow (>1% of TVL)
material = delta.filter(
    pl.col("delta_tvl").is_not_null()
    & (pl.col("net_flow").abs() > 0.01 * pl.col("tvl_total_token0"))
)
if material.height > 0:
    same_sign = material.filter(
        ((pl.col("net_flow") > 0) & (pl.col("delta_tvl") > 0)) |
        ((pl.col("net_flow") < 0) & (pl.col("delta_tvl") < 0))
    ).height
    consistent_pct = same_sign / material.height
    qa_check("net LP flow direction-consistent with TVL delta on flow-active days",
             consistent_pct >= 0.6, severity="warn",
             detail=f"consistent={same_sign}/{material.height} ({consistent_pct:.1%})")
else:
    qa_check("net LP flow direction-consistent with TVL delta on flow-active days", True,
             severity="warn", detail="no material-flow days in window")


🟢 PASS | net LP flow direction-consistent with TVL delta on flow-active days  —  consistent=988/1063 (92.9%)


In [7]:
# Distribution coverage — number of dates with a snapshot
dist_cov = db.read_query(f"""
SELECT countDistinct(date) AS d FROM mart_tvl_distribution_daily_centered
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
""")["d"][0]
qa_check("mart_tvl_distribution_daily_centered: ≥ 30 daily snapshots in window",
         int(dist_cov) >= 30, severity="warn",
         detail=f"distinct_dates={int(dist_cov)}")

🟢 PASS | mart_tvl_distribution_daily_centered: ≥ 30 daily snapshots in window  —  distinct_dates=1798


## §3 — Diagnostic visuals

**QA purpose**: TVL daily evolution + active/out-of-range decomposition reconciles to total.

In [8]:
last_90 = fy.filter(pl.col("date") >= date.fromisoformat(START_DATE_90D))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("TVL active vs out-of-range (stacked)", "Reconciliation residual"))
fig.add_trace(go.Bar(x=last_90["date"].to_list(), y=last_90["tvl_active_token0"].to_list(),       name="Active TVL"),    row=1, col=1)
fig.add_trace(go.Bar(x=last_90["date"].to_list(), y=last_90["tvl_outofrange_token0"].to_list(),   name="OOR TVL"),       row=1, col=1)
fig.add_trace(go.Scatter(x=last_90["date"].to_list(), y=last_90["tvl_total_token0"].to_list(),    name="Total (line)",   line=dict(color="white", width=1)), row=1, col=1)
resid = (last_90["tvl_active_token0"] + last_90["tvl_outofrange_token0"] - last_90["tvl_total_token0"]).to_list()
fig.add_trace(go.Bar(x=last_90["date"].to_list(), y=resid, name="residual"), row=2, col=1)
fig.update_layout(height=520, barmode="stack", title_text="TVL decomposition QA (last 90d)")
fig.show()

**QA purpose**: liquidity distribution snapshot — confirms tick reconstruction is non-empty and centered around current_tick.

In [9]:
latest_dist_d = db.read_query(f"""
SELECT max(date) AS d FROM mart_tvl_distribution_daily_centered WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
""")["d"][0]
dist = db.read_query(f"""
SELECT tick_lower, tick_upper, tick_mid, liquidity, tvl_token0, pct_visible_liquidity, current_tick, relative_tick_mid
FROM mart_tvl_distribution_daily_centered
WHERE lower(pool) = lower('{POOL}') AND date = '{latest_dist_d}'
ORDER BY tick_lower
""")
fig = go.Figure()
fig.add_trace(go.Bar(x=dist["tick_mid"].to_list(), y=dist["tvl_token0"].to_list(), name="tvl_token0"))
fig.add_vline(x=int(dist["current_tick"][0]), line_dash="dash", line_color="orange",
              annotation_text=f"current_tick={int(dist['current_tick'][0])}")
fig.update_layout(height=360, title_text=f"Liquidity distribution — {latest_dist_d}")
fig.show()
print(f"distribution rows: {dist.height}")

distribution rows: 201


**QA purpose / exploration**: TVL distribution *through time*, centered on the current tick — drag the slider or press ▶ Play.
Three granularities: **hourly** (rolling window kept by dbt in `mart_tvl_distribution_hourly_centered`, default last 14 days — widen it with the `hourly_distribution_*` dbt vars), **daily** (last `DIST_DAILY_DAYS`), **weekly** (last daily snapshot of each week, last `DIST_WEEKLY_WEEKS`).
Each snapshot covers ±100 bins around the current tick; `pct_visible_tvl` sums to 100 per snapshot. Frame labels show the spot price; hover a bar for its price and % distance from spot. The y-axis is fixed across frames and capped by `DIST_Y_CAP_PCTL` so a single extreme snapshot doesn't flatten the rest.

In [ ]:
# ─── TVL distribution over time — knobs + data ──────────────────────────────
DIST_METRIC       = "pct_visible_tvl"   # or: "tvl_token0", "pct_visible_liquidity", "liquidity"
DIST_END_DATE     = None                # None = latest available snapshot; e.g. END_DATE to align with the QA window
DIST_DAILY_DAYS   = 90
DIST_WEEKLY_WEEKS = 52
DIST_Y_CAP_PCTL   = 95                  # y-axis = this percentile of per-snapshot peaks (+15%), so one outlier doesn't flatten the rest; None = true max

_DIST_COLS = """current_tick, price_token1_in_token0, relative_tick_mid, is_active_bin,
       tvl_token0, liquidity, pct_visible_tvl, pct_visible_liquidity"""
_DIST_CAP = f"AND toDate({{t}}) <= toDate('{DIST_END_DATE}')" if DIST_END_DATE else ""

def _dist_anchor(table, time_col):
    """Latest snapshot in `table` for POOL, capped at DIST_END_DATE when set (None if empty)."""
    return db.read_query(f"""
        SELECT maxOrNull({time_col}) AS t FROM {table}
        WHERE lower(pool) = lower('{POOL}') {_DIST_CAP.format(t=time_col)}
    """)["t"][0]

# Hourly: the whole rolling window the dbt mart keeps, up to the anchor.
hourly_anchor = _dist_anchor("mart_tvl_distribution_hourly_centered", "hour")
dist_hourly = db.read_query(f"""
SELECT hour AS period, {_DIST_COLS}
FROM mart_tvl_distribution_hourly_centered
WHERE lower(pool) = lower('{POOL}') AND hour <= toDateTime('{hourly_anchor}')
ORDER BY period, relative_tick_mid
""") if hourly_anchor is not None else pl.DataFrame()

# Daily: enough history for both the daily and the weekly view.
daily_anchor = _dist_anchor("mart_tvl_distribution_daily_centered", "date")
_history_days = max(DIST_DAILY_DAYS, DIST_WEEKLY_WEEKS * 7 + 7)
dist_daily_all = db.read_query(f"""
SELECT date AS period, {_DIST_COLS}
FROM mart_tvl_distribution_daily_centered
WHERE lower(pool) = lower('{POOL}')
  AND date > toDate('{daily_anchor}') - {_history_days} AND date <= toDate('{daily_anchor}')
ORDER BY period, relative_tick_mid
""") if daily_anchor is not None else pl.DataFrame()

if dist_daily_all.height:
    _days = dist_daily_all["period"].unique().sort()
    dist_daily = dist_daily_all.filter(pl.col("period").is_in(_days.tail(DIST_DAILY_DAYS).to_list()))
    # Weekly = last daily snapshot of each (Monday-start) week.
    _week_last = (
        _days.to_frame()
        .group_by(pl.col("period").dt.truncate("1w").alias("week_start"))
        .agg(pl.col("period").max())
        .sort("week_start")
        .tail(DIST_WEEKLY_WEEKS)
    )
    dist_weekly = dist_daily_all.join(_week_last.select("period"), on="period")
else:
    dist_daily = dist_weekly = pl.DataFrame()

def _span(df):
    return f"{df['period'].min()} → {df['period'].max()}  ({df['period'].n_unique()} frames)" if df.height else "EMPTY"
print(f"hourly : {_span(dist_hourly)}")
print(f"daily  : {_span(dist_daily)}")
print(f"weekly : {_span(dist_weekly)}")

_DIST_LABELS = {
    "pct_visible_tvl": "% of visible TVL", "pct_visible_liquidity": "% of visible liquidity",
    "tvl_token0": "TVL (token0)", "liquidity": "Liquidity",
}

def plot_dist_slider(df, title, time_fmt, play_ms=200):
    """Animated bar chart of the centered distribution: one frame per snapshot, slider + Play."""
    if df.height == 0:
        print(f"⚠ no snapshots for: {title}")
        return
    pdf = (
        df.sort("period", "relative_tick_mid")
        .with_columns(
            pl.col("period").cast(pl.Datetime),
            pl.when(pl.col("is_active_bin") == 1).then(pl.lit("Active bin")).otherwise(pl.lit("Inactive bin")).alias("bin"),
            # price_token1_in_token0 ∝ 1.0001^(-tick) for any pool
            (pl.col("price_token1_in_token0") * 1.0001 ** (-pl.col("relative_tick_mid"))).alias("bin_price"),
            ((1.0001 ** (-pl.col("relative_tick_mid")) - 1) * 100).alias("pct_from_spot"),
        )
        .to_pandas()
    )
    spot = pdf.groupby("period")["price_token1_in_token0"].transform("first")
    pdf["snapshot"] = pdf["period"].dt.strftime(time_fmt) + "  ·  price " + spot.map("{:,.2f}".format)

    # Fixed y-axis across frames. One extreme snapshot would flatten all the others,
    # so cap at a percentile of per-snapshot peaks; capped bars clip, hover shows true values.
    peaks = pdf.groupby("snapshot", sort=False)[DIST_METRIC].max()
    if DIST_Y_CAP_PCTL is None:
        y_top = float(peaks.max()) * 1.05
    else:
        y_top = float(peaks.quantile(DIST_Y_CAP_PCTL / 100)) * 1.15
        clipped = peaks[peaks > y_top].sort_values(ascending=False)
        if len(clipped):
            print(f"y-axis capped at {y_top:,.2f} — {len(clipped)} snapshot(s) exceed it (hover shows true values): "
                  + ", ".join(f"{k.split('  ·')[0]} = {v:,.2f}" for k, v in clipped.head(5).items()))

    fig = px.bar(
        pdf, x="relative_tick_mid", y=DIST_METRIC, color="bin",
        animation_frame="snapshot", animation_group="relative_tick_mid",
        category_orders={"bin": ["Inactive bin", "Active bin"]},
        color_discrete_map={"Inactive bin": "#9ecae1", "Active bin": "#ef3b2c"},
        hover_data={"bin_price": ":,.2f", "pct_from_spot": ":+.2f", "bin": False},
        labels={
            "relative_tick_mid": "Tick relative to current tick",
            DIST_METRIC: _DIST_LABELS.get(DIST_METRIC, DIST_METRIC),
            "snapshot": "Snapshot", "bin_price": "Bin price", "pct_from_spot": "% from spot",
        },
        title=title,
    )
    x_pad = 10
    fig.update_layout(
        bargap=0, height=520, legend_title_text="",
        xaxis_range=[pdf["relative_tick_mid"].min() - x_pad, pdf["relative_tick_mid"].max() + x_pad],
        yaxis_range=[0, y_top],   # fixed across frames
    )
    fig.update_xaxes(zeroline=True, zerolinewidth=1, zerolinecolor="#111111")
    play = fig.layout.updatemenus[0].buttons[0].args[1]
    play["frame"]["duration"] = play_ms
    play["transition"]["duration"] = 0
    fig.show()

In [ ]:
# Hourly — rolling window maintained by dbt (default: last 14 days)
plot_dist_slider(dist_hourly, "TVL distribution around the current tick — hourly", "%Y-%m-%d %H:%M", play_ms=120)

In [ ]:
plot_dist_slider(dist_daily, f"TVL distribution around the current tick — daily (last {DIST_DAILY_DAYS} days)", "%Y-%m-%d", play_ms=200)

In [ ]:
plot_dist_slider(dist_weekly, f"TVL distribution around the current tick — weekly (last snapshot of each week, {DIST_WEEKLY_WEEKS} weeks)", "%Y-%m-%d", play_ms=300)

**QA purpose**: LP center of mass vs current_tick — large persistent deviation = LP lag, useful as PM signal later but here as a reconstruction diagnostic.

In [10]:
metrics = db.read_query(f"""
SELECT date, current_tick, weighted_mean_relative_tick,
       share_overlapping_50_ticks, share_overlapping_100_ticks, share_overlapping_250_ticks
FROM mart_tvl_distribution_daily_metrics
WHERE lower(pool) = lower('{POOL}')
  AND date >= toDate('{START_DATE_90D}') AND date <= toDate('{END_DATE}')
ORDER BY date
""")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("Liquidity center of mass deviation from current tick", "Concentration around spot"))
fig.add_trace(go.Scatter(x=metrics["date"].to_list(), y=metrics["weighted_mean_relative_tick"].to_list(),
                         name="weighted_mean Δtick", line=dict(color="orange")), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
for col in ["share_overlapping_50_ticks", "share_overlapping_100_ticks", "share_overlapping_250_ticks"]:
    fig.add_trace(go.Scatter(x=metrics["date"].to_list(), y=metrics[col].to_list(), name=col), row=2, col=1)
fig.update_layout(height=480, title_text="Liquidity concentration / LP lag (last 90d)")
fig.show()

**QA purpose**: tick × time × volume heatmap — detect missing tick observations (white columns) or weird hot rows.

In [11]:
heat = db.read_query(f"""
SELECT date, tick, time_seconds, volume_token0_equiv
FROM mart_tick_daily
WHERE lower(pool) = lower('{POOL}')
  AND date >= toDate('{START_DATE_30D}') AND date <= toDate('{END_DATE}')
""")
if heat.height > 0:
    pivot = heat.pivot(values="volume_token0_equiv", index="tick", on="date", aggregate_function="sum").fill_null(0)
    z = pivot.drop("tick").to_numpy()
    fig = go.Figure(go.Heatmap(z=z, x=[c for c in pivot.columns if c != "tick"], y=pivot["tick"].to_list(), colorscale="Viridis"))
    fig.update_layout(height=480, title_text="Tick × date volume heatmap (last 30d)")
    fig.show()
else:
    print("⚠ mart_tick_daily empty in window")


**QA purpose**: LP inflows / outflows daily — sanity-check mints vs burns; large net outflow days should align with TVL drops.

In [12]:
flow_90 = flow.filter(pl.col("date") >= date.fromisoformat(START_DATE_90D))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("Daily mints (token0) / burns (token0)", "Net LP flow (token0)"))
fig.add_trace(go.Bar(x=flow_90["date"].to_list(), y=flow_90["mint_token0"].to_list(),                  name="mints",  marker_color="#22c55e"), row=1, col=1)
fig.add_trace(go.Bar(x=flow_90["date"].to_list(), y=(-flow_90["burn_token0"]).to_list(),               name="burns",  marker_color="#ef4444"), row=1, col=1)
fig.add_trace(go.Bar(x=flow_90["date"].to_list(), y=flow_90["net_lp_flow_token0"].to_list(),           name="net flow", marker_color="#3b82f6"), row=2, col=1)
fig.update_layout(height=480, barmode="relative", title_text="LP flow QA (last 90d)")
fig.show()

## §4 — QA verdict

In [13]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: PASS
  PASS: 6   WARN: 0   FAIL: 0


label,status,severity,detail
str,str,str,str
"""int_range_liquidity_daily: tic…","""PASS""","""fail""","""0"""
"""int_range_liquidity_daily: net…","""PASS""","""fail""","""0"""
"""int_range_liquidity_daily: tic…","""PASS""","""fail""","""0"""
"""TVL active + OOR ≈ total (±1%)""","""PASS""","""warn""","""days_off=0"""
"""net LP flow direction-consiste…","""PASS""","""warn""","""consistent=988/1063 (92.9%)"""
"""mart_tvl_distribution_daily_ce…","""PASS""","""warn""","""distinct_dates=1798"""
